In [35]:
import casadi as ca
import numpy as np

In [36]:
# Données du problème
l, m, mr, md = 7, 5, 2, 3

A = np.array([[-1, -1, 0, 0, 0, 0, 0],
              [0, 0, -1, -1, 0, 0, 0],
              [1, 0, 0, 1, -1, 0, -1],
              [0, 1, 0, 0, 1, -1, 0],
              [0, 0, 1, 0, 0, 1, 1]])

r = np.array([100, 10, 1000, 100, 100, 10, 1000])
p_r = np.array([105, 104])
f_d = np.array([0.08, -1.3, 0.13])

## Problème 1

In [37]:
# Variables de décision
q = ca.SX.sym('q', l)
p = ca.SX.sym('p', m)
f = ca.SX.sym('f', m)

obj = ca.sumsqr(A @ q - f) + ca.sumsqr(r * q * ca.fabs(q) + A.T @ p)

# Contraintes
eq1 = A @ q - f
eq2 = A.T @ p + r * q * ca.fabs(q)
# Contraintes sur les pressions imposées
c_pr = ca.vertcat(p[:mr] - p_r)
# Contraintes sur les flux imposés
c_fd = ca.vertcat(f[mr:] - f_d)

# Optimisation
nlp = {'x': ca.vertcat(q, p, f),
       'f': obj,
       'g': ca.vertcat(eq1, eq2, c_pr, c_fd)}

solver = ca.nlpsol('solver', 'ipopt', nlp)

# Résolution
res = solver(x0=np.zeros(l + m + m),
             lbx=-ca.inf,
             ubx=ca.inf,
             lbg=0,
             ubg=0)

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:       45
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:       66

Total number of variables............................:       17
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:       17
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 1.05e+02 0.00e+00  -1.0 0.00e+00    -  0.00e+00 0.00e+00 

In [38]:
# Extraction des solutions
q_opt = np.array(res['x'][:l].full().flatten())
p_opt = np.array(res['x'][l:l+m].full().flatten())
f_opt = np.array(res['x'][l+m:].full().flatten())

# Vérification des contraintes
f = A @ q_opt
Kirchoff = A.T @ p_opt + r * q_opt * np.abs(q_opt)

print("Débits optimaux q:", q_opt)
print("Pressions optimales p:", p_opt)
print("Bilan de flux (f):", f)
print("Loi de Kirchoff (A^T p + r • q • |q|):", Kirchoff)

Débits optimaux q: [-0.088149 -0.788305 -0.080241 -0.133305 -0.233179  0.278516 -0.068275]
Pressions optimales p: [105.       104.       105.777024 111.214255 110.438544]
Bilan de flux (f): [ 0.876454  0.213546  0.08     -1.3       0.13    ]
Loi de Kirchoff (A^T p + r • q • |q|): [-0. -0. -0. -0. -0.  0. -0.]


In [39]:
# Sauvegarde des résultats du Problème 1
q_opt_1 = q_opt.copy()
p_opt_1 = p_opt.copy()
f_opt_1 = f_opt.copy()

test

In [40]:
# Variables de décision
q = ca.SX.sym('q', l)
p = ca.SX.sym('p', m)
f = ca.SX.sym('f', m)

obj = ca.sumsqr(A @ q - f) + ca.sumsqr(r * q * ca.fabs(q) + A.T @ p)

# Contraintes sur les pressions imposées
c_pr = ca.vertcat(p[:mr] - p_r)
# Contraintes sur les flux imposés
c_fd = ca.vertcat(f[mr:] - f_d)

# Optimisation
nlp = {'x': ca.vertcat(q, p, f),
       'f': obj,
       'g': ca.vertcat(c_pr, c_fd)}

solver = ca.nlpsol('solver', 'ipopt', nlp)

# Résolution
res = solver(x0=np.zeros(l + m + m),
             lbx=-ca.inf,
             ubx=ca.inf,
             lbg=0,
             ubg=0)

# Extraction des solutions
q_opt = np.array(res['x'][:l].full().flatten())
p_opt = np.array(res['x'][l:l+m].full().flatten())
f_opt = np.array(res['x'][l+m:].full().flatten())

# Vérification des contraintes
f = A @ q_opt
Kirchoff = A.T @ p_opt + r * q_opt * np.abs(q_opt)

print("Débits optimaux q:", q_opt)
print("Pressions optimales p:", p_opt)
print("Bilan de flux (f):", f)
print("Loi de Kirchoff (A^T p + r • q • |q|):", Kirchoff)

# Sauvegarde des résultats du Problème 1
q_opt_1 = q_opt.copy()
p_opt_1 = p_opt.copy()
f_opt_1 = f_opt.copy()

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        5
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:       66

Total number of variables............................:       17
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        5
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 1.05e+02 0.00e+00  -1.0 0.00e+00    -  0.00e+00 0.00e+00 

## Problème 2

In [41]:
A_r = A[:mr, :]
A_d = A[mr:, :]

In [42]:
# Conversion explicite des numpy arrays en CasADi DM
r_cas = ca.DM(r)
p_r_cas = ca.DM(p_r)
A_r_cas = ca.DM(A_r)
A_d_cas = ca.DM(A_d)
f_d_cas = ca.DM(f_d)

# Variables de décision
q = ca.SX.sym('q', l)

# Objectif
obj = (1/3) * ca.mtimes(q.T, r_cas * q * ca.fabs(q)) + ca.mtimes(p_r_cas.T, ca.mtimes(A_r_cas, q))

# Contraintes d'égalité
constr = ca.mtimes(A_d_cas, q) - f_d_cas

# Optimisation
nlp = {'x': q,
       'f': obj,
       'g': constr}

solver = ca.nlpsol('solver', 'ipopt', nlp)

# Résolution
res = solver(x0=np.zeros(l),
             lbx=-ca.inf,
             ubx=ca.inf,
             lbg=0,
             ubg=0)

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:       10
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        7

Total number of variables............................:        7
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        3
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 1.30e+00 4.76e-01  -1.0 0.00e+00    -  0.00e+00 0.00e+00 

In [43]:
def reconstruct_pressures(A, r, q_opt, p_r):
    """
    Reconstruit le vecteur des pressions p à partir de la loi pi1 - pi2 = rj qj |qj|,
    en utilisant le fait que le graphe est connexe et que les pressions p_r sont connues aux noeuds de référence.
    """

    p = np.full(m, np.nan)
    p[:len(p_r)] = p_r

    changed = True
    while changed:
        changed = False
        for j in range(l):
            nodes = [i for i in range(m) if A[i, j] != 0]
            i1, i2 = nodes
            delta_p = r[j] * q_opt[j] * abs(q_opt[j])

            known_i1 = not np.isnan(p[i1])
            known_i2 = not np.isnan(p[i2])

            # La relation est : A[i1,j]*p[i1] + A[i2,j]*p[i2] = -delta_p
            if known_i1 and not known_i2:
                # p[i2] = (-delta_p - A[i1,j]*p[i1]) / A[i2,j]
                p[i2] = (-delta_p - A[i1, j]*p[i1]) / A[i2, j]
                changed = True
            elif known_i2 and not known_i1:
                # p[i1] = (-delta_p - A[i2,j]*p[i2]) / A[i1,j]
                p[i1] = (-delta_p - A[i2, j]*p[i2]) / A[i1, j]
                changed = True

    return p

In [44]:
# Extraction des solutions
q_opt = np.array(res['x'].full().flatten())
f_r_opt = A_r @ q_opt
f_opt = np.concatenate((f_r_opt, f_d))
p_opt = reconstruct_pressures(A, r, q_opt, p_r)

In [45]:
# Vérification des contraintes
f = A @ q_opt
Kirchoff = A.T @ p_opt + r * q_opt * np.abs(q_opt)

print("Débits optimaux q:", q_opt)
print("Pressions optimales p:", p_opt)
print("Bilan de flux (f):", f)
print("Loi de Kirchoff (A^T p + r • q • |q|):", Kirchoff)

Débits optimaux q: [-0.088149 -0.788305 -0.080241 -0.133305 -0.233179  0.278516 -0.068275]
Pressions optimales p: [105.       104.       105.777024 111.214255 110.438544]
Bilan de flux (f): [ 0.876454  0.213546  0.08     -1.3       0.13    ]
Loi de Kirchoff (A^T p + r • q • |q|): [-0.  0.  0.  0. -0.  0. -0.]


In [46]:
# Sauvegarde des résultats du Problème 2
q_opt_2 = q_opt.copy()
p_opt_2 = p_opt.copy()
f_opt_2 = f_opt.copy()

## Comparaison des deux méthodes

In [47]:
np.set_printoptions(precision=6, suppress=True)

print("=" * 65)
print(f"{'':>15} {'Problème 1':>15} {'Problème 2':>20}")
print("=" * 65)

# --- Débits q ---
print("\nDébits q (l=7 arêtes) :")
for j in range(l):
    print(f"  q[{j+1}]  {q_opt_1[j]:>22.6f} {q_opt_2[j]:>20.6f}")

# --- Pressions p ---
print("\nPressions p (m=5 noeuds) :")
for i in range(m):
    print(f"  p[{i+1}]  {p_opt_1[i]:>22.6f} {p_opt_2[i]:>20.6f}")

# --- Écart entre les deux solutions ---
print("\n--- Écarts entre les deux solutions ---")
print(f"  ||q1 - q2||_inf = {np.max(np.abs(q_opt_1 - q_opt_2)):.2e}")
print(f"  ||p1 - p2||_inf = {np.max(np.abs(p_opt_1 - p_opt_2)):.2e}")

# --- Résidus physiques ---
kirchhoff_1 = A.T @ p_opt_1 + r * q_opt_1 * np.abs(q_opt_1)
kirchhoff_2 = A.T @ p_opt_2 + r * q_opt_2 * np.abs(q_opt_2)
flux_1 = A @ q_opt_1
flux_2 = A @ q_opt_2

print("\n--- Critères physiques ---")
print(f"  ||A^T p + r q|q|| (Pb1) : {np.linalg.norm(kirchhoff_1):.2e}")
print(f"  ||A^T p + r q|q|| (Pb2) : {np.linalg.norm(kirchhoff_2):.2e}")
print(f"  ||A q - f||      (Pb1) : {np.linalg.norm(flux_1 - f_opt_1):.2e}")
print(f"  ||A q - f||      (Pb2) : {np.linalg.norm(flux_2 - f_opt_2):.2e}")

                     Problème 1           Problème 2

Débits q (l=7 arêtes) :
  q[1]               -0.088149            -0.088149
  q[2]               -0.788305            -0.788305
  q[3]               -0.080241            -0.080241
  q[4]               -0.133305            -0.133305
  q[5]               -0.233179            -0.233179
  q[6]                0.278516             0.278516
  q[7]               -0.068275            -0.068275

Pressions p (m=5 noeuds) :
  p[1]              105.000000           105.000000
  p[2]              104.000000           104.000000
  p[3]              105.777024           105.777024
  p[4]              111.214255           111.214255
  p[5]              110.438544           110.438544

--- Écarts entre les deux solutions ---
  ||q1 - q2||_inf = 9.69e-11
  ||p1 - p2||_inf = 1.71e-09

--- Critères physiques ---
  ||A^T p + r q|q|| (Pb1) : 6.49e-12
  ||A^T p + r q|q|| (Pb2) : 7.75e-09
  ||A q - f||      (Pb1) : 1.10e-11
  ||A q - f||      (Pb2) : 2.22e-